# **Data Mining I | <span style="color: steelblue;">Segmentation</span>**

## A Clustering-Based Exploration of WorkplaceAbsence Patterns

*Group 07*

*Group members* : Francisco Gomes (20221810), Margarida Marchão (20221901), Pedro Coimbras (20211573) and Marta Alves (20221890).

In [ ]:
# ==============================
# Core scientific stack
# ==============================
import numpy as np
import pandas as pd

# ==============================
# Visualization
# ==============================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors
import seaborn as sns
import plotly.express as px

from pylab import rcParams
rcParams['figure.figsize'] = (30, 15)

# ==============================
# Clustering algorithms
# ==============================
from sklearn.cluster import KMeans, AgglomerativeClustering

# ==============================
# Preprocessing & scaling
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# ==============================
# Dimensionality reduction (visualization only)
# ==============================
import umap

# ==============================
# Clustering evaluation & hierarchy
# ==============================
from sklearn.metrics import confusion_matrix
from scipy.cluster.hierarchy import dendrogram, linkage

# ==============================
# Utilities
# ==============================
from tqdm.auto import tqdm
import warnings
from pathlib import Path

# ==============================
# Project-specific functions
# ==============================
from modeling_pipeline import (
    calculate_group_means,
    create_dendrogram,
    visualize_inertia_silhouette,
    plot_dimensionality_reduction_results,
    plot_r2_hc
)

# ==============================
# Global settings
# ==============================å
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)


In [ ]:
base_path = Path.cwd()

# Go one folder back and into 'data'
data_path = base_path.parent / "data"

# Read the CSV
worker = pd.read_csv(
    data_path / "Processed Absenteeism Dataset.csv"
)

display(worker.head())

,Days since previous absence,Transportation expense,Distance from Residence to Work,Estimated commute time,Service time,Years until retirement,Date of Birth,Disciplinary failure,Number of children,Social drinker,Social smoker,Number of pets,Weight,Height,Absenteeism time in hours,Day of the week_Monday,Day of the week_Saturday,Day of the week_Thursday,Day of the week_Tuesday,Day of the week_Wednesday,Seasons_Spring,Seasons_Summer,Seasons_Winter,Education_2,Education_3,Education_4,bmi_category_Obesity Class I,bmi_category_Obesity Class II,bmi_category_Overweight,Month_sin,Month_cos,Reason_for_abs_freq
0,0.0,289,36,69,13,32,1992-08-15,No,2,Y,No,1,90,172,4,False,False,False,True,False,False,True,False,False,False,False,True,False,False,-0.5,-0.866025,0.04125
1,0.0,118,13,26,18,15,1975-09-02,Yes,1,Y,No,0,98,178,0,False,False,False,True,False,True,False,False,False,False,False,True,False,False,-0.5,-0.866025,0.05375
2,0.0,179,51,108,18,27,1987-04-08,No,0,Yes,No,0,89,170,2,False,False,False,False,True,False,True,False,False,False,False,True,False,False,-0.5,-0.866025,0.26125
3,0.0,279,5,5,14,26,1986-07-25,No,2,Yes,Yes,0,68,168,4,False,False,True,False,False,True,False,False,False,False,False,False,False,False,-0.5,-0.866025,0.01875
4,0.0,289,36,69,13,32,1992-08-15,No,2,Yes,No,1,90,172,2,False,False,True,False,False,False,True,False,False,False,False,True,False,False,-0.5,-0.866025,0.26125


In [ ]:
worker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 32 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Days since previous absence      800 non-null    float64
 1   Transportation expense           800 non-null    int64  
 2   Distance from Residence to Work  800 non-null    int64  
 3   Estimated commute time           800 non-null    int64  
 4   Service time                     800 non-null    object 
 5   Years until retirement           800 non-null    int64  
 6   Date of Birth                    800 non-null    object 
 7   Disciplinary failure             800 non-null    object 
 8   Number of children               800 non-null    int64  
 9   Social drinker                   800 non-null    object 
 10  Social smoker                    800 non-null    object 
 11  Number of pets                   800 non-null    int64  
 12  Weight                

# **Scaling**

Clustering algorithms based on distance metrics are highly sensitive to feature scale.  
Given the heterogeneous nature of the variables in this dataset (e.g., demographic, behavioral, and absenteeism-related measures), different scaling strategies were evaluated to assess their impact on cluster structure and stability.

The following approaches are considered:

- **No scaling**:  
  Used as a baseline to understand how raw feature magnitudes influence clustering results.

- **Standard Scaling (Z-score normalization)**:  
  Centers variables to zero mean and unit variance, ensuring that all features contribute equally to Euclidean distance calculations.

- **Min–Max Scaling**:  
  Rescales features to a fixed range \([0, 1]\), preserving relative distances while reducing the influence of extreme values.

- **Robust Scaling**:  
  Scales features using median and interquartile range, making it less sensitive to outliers.

Each scaled dataset is analyzed independently in the clustering stage to evaluate the robustness of the resulting segmentation and to ensure that conclusions are not driven by a particular preprocessing choice.



In [ ]:
# No scaling
worker_no_scl = worker.copy()
# StandardScaler
worker_st_scl = pd.DataFrame(StandardScaler().fit_transform(worker),columns=worker.columns,index=worker.index,)
# MinMaxScaler
worker_mm_scl = pd.DataFrame(MinMaxScaler().fit_transform(worker),columns=worker.columns,index=worker.index,)
# RobustScaler
worker_rb_scl = pd.DataFrame(RobustScaler().fit_transform(worker),columns=worker.columns,index=worker.index,)

ValueError: could not convert string to float: '-'

# **Clustering**

In the notebook, the cluster analysis is grouped by data inputs: no scaling,standard scaler, minmax scaler, and robust scaler. In each group, the following methods of clustering were used and are presented in this order:

1. KMeans
2. Ward (Hierarchical)

Given that the Ward method consistently outperforms the other methods in terms of the R2 metric across all cluster numbers, it is justifiable to focus on analyzing the Ward method for hierarchical clustering in this context. The Ward method provides a superior explanation of the variance within the data, making it the most reliable choice for this analysis.

In [ ]:
plot_r2_hc(worker_no_scl, max_nclus=12)

ValueError: Input X contains NaN.
AgglomerativeClustering does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
visualize_inertia_silhouette(worker_no_scl)

In [ ]:
kmeans = KMeans(n_clusters = 5, random_state = 0).fit(worker_no_scl)
worker['no_kmeans5'] = kmeans.predict(worker_no_scl)

In [ ]:
calculate_group_means(worker, 'no_kmeans5')

0. **Generalist Mid-Age Workforce**

1. **Veteran Local Employees**

2. **Lifestyle-Constrained Young Employees**

3. **Stable Professional Core**

4. **Physically Strained High-Risk Employees**


In [ ]:
create_dendrogram(worker_no_scl, 'ward')

In [ ]:
worker['no_ward5'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 5
    ).fit_predict(worker_no_scl)

In [ ]:
calculate_group_means(worker, 'no_ward5')

0. **Veteran Local High-Absence Employees**

1. **Stable Long-Distance Professionals**

2. **Young Lifestyle-Constrained Employees**

3. **Family-Oriented Social Workforce**

4. **Balanced Mid-Career Employees**


In [ ]:
plot_r2_hc(worker_st_scl, max_nclus=12)

In [ ]:
visualize_inertia_silhouette(worker_st_scl)

In [ ]:
kmeans = KMeans(n_clusters = 8, random_state = 0).fit(worker_st_scl)
worker['st_kmeans8'] = kmeans.predict(worker_st_scl)

In [ ]:
calculate_group_means(worker, 'st_kmeans8')

0. **Veteran Physically Strained Employees**

1. **Local Overweight Stable Workforce**

2. **Young Highly Educated Professionals**

3. **Seasonal Long-Term Employees**

4. **Saturday-Oriented Low-Absence Workers**

5. **Young Pet-Oriented Lifestyle Employees**

6. **Experienced Reliable Workforce**

7. **Family-Oriented Commuter Employees**


In [ ]:
create_dendrogram(worker_st_scl, 'ward')

In [ ]:
worker['st_ward8'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 8
    ).fit_predict(worker_st_scl)

In [ ]:
calculate_group_means(worker, 'st_ward8')

0. **High-Absence Physically Strained Employees**

1. **Stable Local Overweight Workforce**

2. **Family-Oriented Long-Distance Employees**

3. **Low-Risk Young Stable Employees**

4. **Moderate Absenteeism Balanced Workforce**

5. **Severely Obese High-Risk Employees**

6. **Young Highly Educated Professionals**

7. **Saturday-Oriented Young Workforce**


In [ ]:
plot_r2_hc(worker_mm_scl, max_nclus=12)

In [ ]:
visualize_inertia_silhouette(worker_mm_scl)

In [ ]:
kmeans = KMeans(n_clusters = 6, random_state = 0).fit(worker_mm_scl)
worker['mm_kmeans6'] = kmeans.predict(worker_mm_scl)

In [ ]:
calculate_group_means(worker, 'mm_kmeans6')

0. **Socially Active Mid-Career Employees**

1. **Stable Winter-Oriented Workforce**

2. **Seasonal Summer Commuters**

3. **Young Educated Low-Risk Professionals**

4. **Physically Strained High-BMI Employees**

5. **Young Pet-Oriented Reliable Employees**


In [ ]:
create_dendrogram(worker_mm_scl, 'ward')

In [ ]:
worker['mm_ward8'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 8
    ).fit_predict(worker_mm_scl)

In [ ]:
calculate_group_means(worker, 'mm_ward8')

0. **Obese Senior Long-Tenure Employees**

1. **Spring-Oriented Family Workforce**

2. **Seasonal Summer Commuters**

3. **Overweight Long-Distance Professionals**

4. **Highly Educated Lightweight Employees**

5. **Socially Active Mid-Career Employees**

6. **Young Pet-Oriented Workforce**

7. **Stable Local Overweight Employees**


In [ ]:
plot_r2_hc(worker_rb_scl, max_nclus=12)

In [ ]:
visualize_inertia_silhouette(worker_rb_scl)

In [ ]:
kmeans = KMeans(n_clusters = 4, random_state = 0).fit(worker_rb_scl)
worker['rb_kmeans4'] = kmeans.predict(worker_rb_scl)

In [ ]:
calculate_group_means(worker, 'rb_kmeans4')

0. **Balanced Mid-Career Workforce**

1. **Young Overweight Local Employees**

2. **Young Pet-Oriented Low-Absence Employees**

3. **Older High-Absenteeism Workforce**


In [ ]:
create_dendrogram(worker_rb_scl, 'ward')

In [ ]:
worker['rb_ward9'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 9
    ).fit_predict(worker_rb_scl)

In [ ]:
worker['rb_ward10'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 10
    ).fit_predict(worker_rb_scl)

In [ ]:
worker['rb_ward11'] = AgglomerativeClustering(
    linkage = 'ward', n_clusters = 11
    ).fit_predict(worker_rb_scl)

In [ ]:
calculate_group_means(worker, 'rb_ward10')

0. **Mid-Age Stable Employees**

1. **Employees with Chronic Absenteeism**

2. **Young Low-Burned Preformers**

3. **Pet Lovers Employees**

4. **Long-Distance Commuters with Pet Constraints**

5. **Early-Career Unstable Workers**

6. **Physically Strained Operational Staff**

7. **Socially Driven High-Absence Group**

8. **Healthy Young Reliable Employees**

9. **Veteran Reliable Workforce**

----


# **Dimensionality Reduction for Cluster Visualization (UMAP)**

To facilitate the visual inspection of the clustering results, a non-linear dimensionality reduction technique was applied to project the high-dimensional feature space into two dimensions.

Uniform Manifold Approximation and Projection (UMAP) was used exclusively for **visualization purposes**, allowing the preservation of local neighborhood structure while providing an intuitive representation of cluster separation. Importantly, dimensionality reduction was **not applied prior to clustering**, ensuring that cluster formation remained grounded in the original feature space.

The UMAP embedding is therefore used to visually assess cluster cohesion and separation for the selected clustering solutions.


In [ ]:
umap_object = umap.UMAP(n_neighbors = 30, min_dist = 0.15, random_state = 42)

In [ ]:
umap_embedding = umap_object.fit_transform(worker_rb_scl)

In [ ]:
plot_dimensionality_reduction_results(umap_embedding, worker['rb_ward10'])

# **Final Cluster Labeling and Segment Distribution**

Following the selection of the final clustering solution, numerical cluster identifiers were mapped to descriptive and interpretable segment labels. This step enables clearer communication of results and facilitates downstream analysis and interpretation.

The final cluster assignments are stored in a dedicated column, and the distribution of observations across segments is examined to assess the relative size and representativeness of each employee group.


In [ ]:
clusters_mapping = {
    0: "Mid-Age Stable Employees",
    1: "Employees with Chronic Absenteeism",
    2: "Young Low-Burden Performers",
    3: "Pet Lovers Employees",
    4: "Long-Distance Commuters with Pet Constraints",
    5: "Early-Career Unstable Workers",
    6: "Physically Strained Operational Staff",
    7: "Socially Driven High-Absence Group",
    8: "Healthy Young Reliable Employees",
    9: "Veteran Reliable Workforce"
}

In [ ]:
# Create the final solution column
worker['final_solution'] = worker['rb_ward10']
# Map the cluster numbers to names
worker['Final_Cluster'] = [clusters_mapping[value] for value in worker['final_solution']]

In [ ]:
worker[['Final_Cluster']].value_counts()

# **Univariate Distribution Analysis by Cluster**

To further characterize and interpret the identified employee segments, univariate distributions of key variables are examined across the final clusters. Histograms with kernel density estimates are used to compare how each feature is distributed within and across clusters.

This analysis supports the qualitative interpretation of clusters by highlighting differences in demographic, behavioral, and absenteeism-related variables, and helps validate the coherence and distinctiveness of the final segmentation.

In [ ]:
cluster_colors = {
    0: 'blue',          
    1: 'orange',      
    2: 'green',      
    3: 'red',       
    4: 'purple',      
    5: 'brown',    
    6: 'pink',      
    7: 'grey',  
    8: 'yellow',  
    9: 'deepskyblue'   
}

cols_to_plot = [
    'Age',
    'Transportation expense',
    'Distance from Residence to Work',
    'Service time',
    'Years until retirement',
    'Disciplinary failure',
    'Number of children',
    'Social drinker',
    'Social smoker',
    'Number of pets',
    'Weight',
    'Height',
    'Absenteeism time in hours',
    'Reason_for_abs_freq',
    'Final_Cluster'
]


In [ ]:
# Loop through all columns except numeric cluster labels
for col in tqdm(cols_to_plot):
    sns.histplot(
        data=worker.dropna(subset=[col]),
        x=col,
        hue='final_solution',
        kde=True,
        palette=cluster_colors,
        hue_order=sorted(cluster_colors.keys()),
        legend=True
    )

    plt.title(f"Distribution of {col} by Cluster")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()